In [ ]:
import os
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")

In [ ]:
import os, textwrap

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip install -q -r requirements.txt
!apt-get install -y -q gdal-bin

In [ ]:
# Patch py_tools_ds bool-mask bug (needed for AROSICS)
import importlib
import numpy as np
from py_tools_ds.geo.raster import reproject as reproj

importlib.reload(reproj)
_orig = reproj.warp_ndarray

def _warp_ndarray_mask_safe(ndarray, *args, **kwargs):
    for k in ("in_nodata", "out_nodata"):
        if k in kwargs and isinstance(kwargs[k], (bool, np.bool_)):
            kwargs[k] = int(kwargs[k])
    if isinstance(ndarray, np.ndarray) and ndarray.dtype == np.bool_:
        ndarray = ndarray.astype(np.uint8)
        kwargs.setdefault("in_nodata", 0)
        kwargs.setdefault("out_nodata", 0)
        kwargs.pop("out_dtype", None)
        kwargs.setdefault("rspAlg", "near")
    return _orig(ndarray, *args, **kwargs)

reproj.warp_ndarray = _warp_ndarray_mask_safe

In [ ]:
import os, shutil
from pathlib import Path
from datetime import datetime
from pprint import pprint

import pandas as pd
from tqdm import tqdm

from documentation.config import PipelineConfig
from documentation.pairs_artifacts import (
    load_aois_catalogue, get_aoi_by_index, load_pipeline_config,
    AoiPaths,
    make_pair_id, RunPaths, TileRecord,
    load_pairs_sorted, write_aoi_pairs_csv, pick_pair_by_rank,
    register_pair, registry_has_pair, save_run_lock,
    write_emit_metadata, write_s2_metadata,
    write_tile_metadata, write_manifest_csv, write_global_manifest,
    write_archive_map, tif_geo_summary, copy_any,
)
from documentation.report_builder import (
    ReportBuilder, plot_side_by_side_rgb, plot_realignment_summary,
)
from data.pairing.pairs_utils import point_buffer_bbox
from data.pairing.pairing import pair_emit_to_s2_batched
from data.EMIT.emit_search import search, download_reflectance, refetch_emit_pick
from data.EMIT.emit_utils import (
    load_emit_wavelengths_nm_from_nc, cache_wavelengths_json, closest_bands,
)
from data.S2.s2_search import download_s2_spectral_stack, fetch_s2_item_by_id
from geometry.alignment import align_emit_s2_pair
from tiles_helpers.utils import (
    find_valid_paired_tiles, save_tile_pair,
    plot_tile_pair_simple, write_emit_b32_tile,
)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import earthaccess as ea
ea.login(strategy="environment", persist=True)

In [ ]:
PAIR_RANK   = 0
AOI_INDICES = list(range(0, 500))
DRIVE_ROOT  = Path("/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches")
RUN_TAG     = datetime.now().strftime("%Y-%m-%d")

In [ ]:
drive_base = DRIVE_ROOT / RUN_TAG
drive_base.mkdir(parents=True, exist_ok=True)

for seed in ("aois.csv", "pipeline_config.yaml"):
    dst = drive_base / seed
    if not dst.exists():
        shutil.copy(seed, dst)
        print(f"Seeded {dst}")

catalogue = load_aois_catalogue(drive_base)
config_dict = load_pipeline_config(drive_base)

if AOI_INDICES is None:
    AOI_INDICES = list(range(len(catalogue)))

print(f"{len(AOI_INDICES)} AOIs selected, PAIR_RANK={PAIR_RANK}")
print(f"Drive root: {drive_base}")
print("\nFull config from pipeline_config.yaml:")
pprint(config_dict)

In [ ]:
def _cleanup_empty_aoi(aoi):
    if not aoi.root.exists():
        return
    children = list(aoi.root.iterdir())
    has_pair_dirs = any(c.is_dir() for c in children)
    if not has_pair_dirs:
        shutil.rmtree(aoi.root, ignore_errors=True)


results = []

for aoi_idx in tqdm(AOI_INDICES, desc="AOIs"):
    aoi_row = get_aoi_by_index(drive_base, aoi_idx)
    aoi_label = f"[{aoi_idx:03d}] {aoi_row['name']}"

    config = PipelineConfig.from_dict({
        **config_dict,
        "aoi_lat":    aoi_row["lat"],
        "aoi_lon":    aoi_row["lon"],
        "drive_root": str(drive_base),
    })

    ROOT = Path(config.local_root)
    ROOT.mkdir(parents=True, exist_ok=True)
    aoi = AoiPaths.build(drive_base, config.aoi_lat, config.aoi_lon)
    config.to_json(aoi.config_json)

    try:
        aoi_geom = point_buffer_bbox(config.aoi_lon, config.aoi_lat, config.aoi_buffer_m)
        search_start, search_end = config.search_bounds

        picks = search(
            bbox=aoi_geom.bounds,
            start=search_start,
            end=search_end,
            cloud_cover=(0.0, config.max_emit_cloud_pct),
        )
        if not picks:
            print(f"  {aoi_label}: no EMIT granules found — skipping")
            results.append({"aoi_idx": aoi_idx, "name": aoi_row["name"], "status": "no_emit"})
            _cleanup_empty_aoi(aoi)
            continue

        for old in ROOT.glob("pairs_batch_*.jsonl"):
            old.unlink()

        pair_emit_to_s2_batched(
            emit_items       = picks,
            aoi_geom_wgs84   = aoi_geom,
            out_dir          = ROOT,
            s2_api           = config.s2_api,
            s2_collection    = config.s2_collection,
            days             = config.pairing_days,
            window_days      = config.pairing_window_days,
            resume           = False,
            emit_top_n_per_day = config.emit_top_n_per_day,
            emit_max_cloud_pct = config.max_emit_cloud_pct,
            scl_cloud_max    = config.max_s2_cloud_frac,
            top_k_prefilter  = config.top_k_prefilter,
            tile_m           = config.pairing_tile_m,
            sun_deg_max      = config.sun_deg_max,
            max_tod_diff_h   = config.max_tod_diff_h,
        )

        pairs = load_pairs_sorted(ROOT, sort_by="expected_clear_km2")
        write_aoi_pairs_csv(aoi, pairs, config_dict=config.to_dict())

        if PAIR_RANK >= len(pairs):
            print(f"  {aoi_label}: only {len(pairs)} pairs, rank {PAIR_RANK} unavailable — skipping")
            results.append({"aoi_idx": aoi_idx, "name": aoi_row["name"], "status": "insufficient_pairs", "n_pairs": len(pairs)})
            _cleanup_empty_aoi(aoi)
            continue

        pair = pick_pair_by_rank(pairs, rank=PAIR_RANK)

        if registry_has_pair(aoi, pair["emit_granuleur"], pair["s2_id"],
                             config_fingerprint=config.run_fingerprint):
            print(f"  {aoi_label}: rank {PAIR_RANK} already completed — skipping")
            results.append({"aoi_idx": aoi_idx, "name": aoi_row["name"], "status": "already_done"})
            continue

    except Exception as e:
        print(f"  {aoi_label}: pairing failed — {e}")
        results.append({"aoi_idx": aoi_idx, "name": aoi_row["name"], "status": "pairing_error", "error": str(e)})
        _cleanup_empty_aoi(aoi)
        continue

    try:
        emit_pick = refetch_emit_pick(pair["emit_granuleur"])
        s2_item   = fetch_s2_item_by_id(pair["s2_id"], stac_api=config.s2_api, collection=config.s2_collection)

        emit_paths = download_reflectance(emit_pick, ROOT / "emit", assets=["_RFL_"], download_obs=True)
        emit_nc    = Path(emit_paths[0])
        obs_path   = emit_paths[-1] if len(emit_paths) > 1 else None
        s2_stack, scl_path = download_s2_spectral_stack(s2_item, ROOT / "s2", return_scl=True)

    except Exception as e:
        print(f"  {aoi_label}: download failed — {e}")
        results.append({"aoi_idx": aoi_idx, "name": aoi_row["name"], "status": "download_error", "error": str(e)})
        _cleanup_empty_aoi(aoi)
        continue

    wl_nm = load_emit_wavelengths_nm_from_nc(str(emit_nc))
    cache_wavelengths_json(wl_nm, Path(emit_nc).with_suffix(".wavelengths.json"))

    pair_id = make_pair_id(emit_nc, s2_item.to_dict())
    paths = RunPaths.build(
        emit_nc=emit_nc, local_root=ROOT / pair_id,
        aoi=aoi, pair_id=pair_id, s2_item=s2_item,
    )

    run_entry = register_pair(
        aoi, pair=pair, pair_id=pair_id,
        config_fingerprint=config.run_fingerprint, status="started",
    )

    report = ReportBuilder(
        paths.drive_report_md,
        html_path=paths.drive_report_md.with_suffix(".html"),
        mode="overwrite",
    )
    report.start(title=f"EMIT / S2 pair: {paths.pair_id}")
    report.add_config_table(config)
    config.to_json(paths.drive_meta / "pipeline_config.json")

    emit_summary = write_emit_metadata(emit_pick, paths.drive_meta, report=report)
    s2_summary   = write_s2_metadata(s2_item, paths.drive_meta, report=report)

    try:
        alignment = align_emit_s2_pair(
            emit_nc=emit_nc, s2_stack=s2_stack,
            out_dir=paths.local_alignment, config=config,
            wl_nm=wl_nm, emit_obs_nc=obs_path,
            report=report, keep_intermediate=False,
            emit_info_save_path=paths.drive_alignment / "emit_conversion.json",
        )
        if not alignment.success:
            raise RuntimeError(f"Co-registration failed: {alignment.coreg_info}")

    except Exception as e:
        print(f"  {aoi_label}: alignment failed — {e}")
        register_pair(aoi, pair=pair, pair_id=pair_id,
                      run_uid=run_entry["run_uid"],
                      config_fingerprint=config.run_fingerprint, status="alignment_failed")
        results.append({"aoi_idx": aoi_idx, "name": aoi_row["name"], "status": "alignment_error", "error": str(e)})
        _cleanup_empty_aoi(aoi)
        continue

    envi_bin_trimmed   = alignment.emit_envi_trimmed
    out_s2_tif_trimmed = alignment.s2_tif_trimmed
    envi_bin           = alignment.emit_envi_full

    rgb_before = plot_side_by_side_rgb(
        s2_path=alignment.s2_overlap_path or s2_stack,
        emit_path=envi_bin,
        out_path=paths.drive_plots / "comparison_before_coreg.png",
        wl_nm=wl_nm, title="Before co-registration",
    )
    report.add_image("S2 vs EMIT — Before Co-registration", rgb_before)

    rgb_after = plot_side_by_side_rgb(
        s2_path=out_s2_tif_trimmed,
        emit_path=envi_bin_trimmed,
        out_path=paths.drive_plots / "comparison_after_coreg.png",
        wl_nm=wl_nm, title="After co-registration",
    )
    report.add_image("S2 vs EMIT — After Co-registration", rgb_after)

    _b32_idx_0based, _ = closest_bands(wl_nm, list(config.emit_target_wavelengths_nm))
    _emit_check_bands = [i + 1 for i in sorted(set(_b32_idx_0based))]
    valid_tiles = find_valid_paired_tiles(
        emit_path=str(envi_bin_trimmed),
        s2_path=str(out_s2_tif_trimmed),
        scl_path=scl_path,
        emit_check_bands=_emit_check_bands,
        **config.tile_params,
    )

    if not valid_tiles:
        print(f"  {aoi_label}: 0 valid tiles — skipping")
        register_pair(aoi, pair=pair, pair_id=pair_id,
                      run_uid=run_entry["run_uid"],
                      config_fingerprint=config.run_fingerprint, status="no_tiles")
        results.append({"aoi_idx": aoi_idx, "name": aoi_row["name"], "status": "no_tiles"})
        _cleanup_empty_aoi(aoi)
        continue

    tile_records = []
    emit_b32_idx = None

    for tile_info in valid_tiles:
        k = int(tile_info["idx"])

        emit_out, s2_out, realign_stats = save_tile_pair(
            emit_path=str(envi_bin_trimmed),
            s2_path=str(out_s2_tif_trimmed),
            tile_info=tile_info,
            out_dir=paths.drive_tiles,
            pair_id=paths.pair_id,
            emit_wavelengths_nm=wl_nm,
            realign=True,
            realign_max_shift=1.0,
        )

        emit_out_b32, emit_b32_idx = write_emit_b32_tile(
            emit_out, num_keep=config.spectral_num_keep_bands,
            idx_0based=emit_b32_idx,
            overwrite=True, wavelengths_nm=wl_nm,
            target_wavelengths_nm=config.emit_target_wavelengths_nm,
        )

        plot_png = paths.drive_plots / f"{paths.pair_id}_tile{k:03d}_pair.png"
        plot_tile_pair_simple(
            emit_tile_path=str(emit_out), s2_tile_path=str(s2_out),
            title_suffix=f"tile {k:03d}",
            save_path=str(plot_png), show=False,
            emit_wavelengths_nm=wl_nm,
        )

        rec = TileRecord(
            idx=k, emit_tif=str(emit_out), s2_tif=str(s2_out),
            pair_id=paths.pair_id, plot_png=str(plot_png),
            emit_black_frac=tile_info.get("emit_black_frac"),
            s2_black_frac=tile_info.get("s2_black_frac"),
        )
        rec.emit_geo = tif_geo_summary(emit_out)
        rec.s2_geo   = tif_geo_summary(s2_out)
        if realign_stats is not None:
            rec.realign_stats = realign_stats
        rec.emit_b32_tif = str(emit_out_b32)
        rec.emit_b32_indices_0based = emit_b32_idx.tolist()

        write_tile_metadata(
            rec, tile_info=tile_info, out_dir=paths.drive_tile_meta,
            emit_granule=emit_summary.get("granule_ur"),
            emit_time=emit_summary.get("time"),
            s2_id=s2_summary.get("id"),
            s2_datetime=s2_summary.get("datetime"),
            params=config.tile_params,
        )
        tile_records.append(rec)

    write_manifest_csv(paths.drive_manifest_csv, tile_records)

    realign_plot = plot_realignment_summary(
        tile_records,
        paths.drive_plots / "realignment_shifts.png",
        title=f"Realignment — {paths.pair_id}",
    )
    report.add_realignment_section(tile_records, plot_path=realign_plot)

    write_global_manifest(aoi)

    copy_any(
        paths.local_alignment, paths.drive_alignment,
        overwrite=True, exclude=["tmp", "*.aux.xml", "*.ovr"],
    )
    copy_any(emit_nc,  paths.drive_emit / Path(emit_nc).name,  overwrite=True)
    copy_any(s2_stack, paths.drive_s2   / Path(s2_stack).name, overwrite=True)

    archive_map = {
        "local_ROOT":               str(ROOT),
        "drive_pair_folder":        str(drive_base),
        "drive_raw_emit":           str(paths.drive_emit),
        "drive_raw_s2":             str(paths.drive_s2),
        "drive_emit_reprojections": str(paths.drive_alignment),
    }
    write_archive_map(paths.drive_meta / "archive_map.json", archive_map, report=report)
    report.finalise_html(title=f"Run {paths.run_id}")

    register_pair(
        aoi, pair=pair, pair_id=pair_id,
        run_uid=run_entry["run_uid"],
        config_fingerprint=config.run_fingerprint, status="completed",
    )
    save_run_lock(
        paths.drive_root, pair=pair, pair_id=pair_id,
        run_uid=run_entry["run_uid"],
        config_fingerprint=config.run_fingerprint,
        config_dict=config.to_dict(),
    )

    n_tiles = len(tile_records)
    print(f"  {aoi_label}: {n_tiles} tiles saved")
    results.append({
        "aoi_idx": aoi_idx, "name": aoi_row["name"],
        "status": "completed", "n_tiles": n_tiles,
        "pair_id": pair_id,
    })

    shutil.rmtree(ROOT, ignore_errors=True)

print(f"\nFinished processing {len(AOI_INDICES)} AOIs.")

In [ ]:
df = pd.DataFrame(results)
print(df["status"].value_counts().to_string())
print(f"\nTotal tiles extracted: {df['n_tiles'].sum():.0f}")
df.to_csv(drive_base / f"extraction_summary_rank{PAIR_RANK}.csv", index=False)
df